# Porównanie cech — kategoria i słowa

**Hipoteza:** tekst połączony z deklarowaną kategorią pomaga wykrywać
niespójności. Porównujemy regresję logistyczną z tą gałęzią oraz bez niej.
Pozostałe cechy, parametry, wiersze i pięć foldów sprzedawców są takie same.

W obu wersjach pozostaje parser ilości v1. To porównanie jest oddzielne
od eksperymentu ilościowego drugiej osoby. Model referencyjny pochodzi
z notebooka organizatorów. Każdy uczony krok dopasowujemy w treningu foldu.

In [13]:
import os
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[variable] = '4'

import hashlib
import json
import platform
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import threadpoolctl
from threadpoolctl import threadpool_limits
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
DATA_DIR = None
FOLDS_PATH = None
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXPECTED_TRAIN_SHA256 = 'aefc970e5efc4435cecfe76b901149fdd9ffd06bc62a3f62773379902b34195c'
EXPECTED_FOLDS_SHA256 = 'c6a234966e2b3107f3c24e9d3aa0d565430027cb6e16212ebf612a993ccc6956'

def find_data():
    if DATA_DIR is not None:
        candidates = [Path(DATA_DIR)]
    else:
        candidates = [p for p in [Path('data/public'), Path('data'), Path('.')]
                      if all((p / name).is_file() for name in
                             ['train.csv', 'test.csv', 'sample_submission.csv'])]
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('train.csv')
                           if (p.parent / 'test.csv').is_file()
                           and (p.parent / 'sample_submission.csv').is_file()]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) != 1:
        raise ValueError('Dodaj dane konkursu do Input lub ustaw DATA_DIR.')
    return candidates[0]

def canonical_folds(train):
    values = [int(hashlib.sha256(f'{SEED}:{seller}'.encode()).hexdigest()[:16], 16) % 5 + 1
              for seller in train.seller_id.astype(str)]
    return pd.DataFrame({'id': train.id, 'seller_id': train.seller_id, 'fold': values})

def load_folds(train):
    output = OUTPUT_ROOT / 'shared' / 'folds.csv'
    output.parent.mkdir(exist_ok=True)
    if FOLDS_PATH is not None:
        fold_source = Path(FOLDS_PATH)
    elif Path('folds.csv').is_file():
        fold_source = Path('folds.csv')
    elif output.is_file():
        fold_source = output
    else:
        # Na Kaggle pierwszy notebook zapisze ten sam plik co w pakiecie.
        canonical_folds(train).to_csv(output, index=False, lineterminator='\n')
        fold_source = output
    table = pd.read_csv(fold_source)
    assert {'id', 'seller_id', 'fold'}.issubset(table.columns), 'Niepełny plik foldów.'
    assert table.id.is_unique and set(table.id) == set(train.id), 'Inne ID w foldach.'
    table = table.set_index('id').loc[train.id].reset_index()
    assert table.seller_id.tolist() == train.seller_id.tolist(), 'Inni sprzedawcy w foldach.'
    assert table.fold.isin([1, 2, 3, 4, 5]).all(), 'Nieprawidłowe numery foldów.'
    table['fold'] = table.fold.astype(int)
    assert table.groupby('seller_id').fold.nunique().eq(1).all(), 'Sprzedawca w kilku foldach.'
    assert table['fold'].tolist() == canonical_folds(train)['fold'].tolist(), 'Podział różni się od uzgodnionego startera.'
    encoded = table[['id', 'seller_id', 'fold']].to_csv(index=False, lineterminator='\n').encode()
    digest = hashlib.sha256(encoded).hexdigest()
    assert digest == EXPECTED_FOLDS_SHA256, 'Hash foldów nie pasuje do uzgodnionej edycji.'
    output.write_bytes(encoded)
    return table, digest

data_dir = find_data()
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample = pd.read_csv(data_dir / 'sample_submission.csv')
assert hashlib.sha256((data_dir / 'train.csv').read_bytes()).hexdigest() == EXPECTED_TRAIN_SHA256, 'Inna edycja treningu.'
assert train.target.isin([0, 1]).all()
assert train.id.is_unique and test.id.is_unique
assert set(train.id).isdisjoint(test.id)
assert set(train.seller_id).isdisjoint(test.seller_id)
assert sample.columns.tolist() == ['id', 'target']
assert sample.id.tolist() == test.id.tolist()
fold_table, fold_sha256 = load_folds(train)
fold_ids = fold_table.fold.to_numpy()
splits = [(np.flatnonzero(fold_ids != fold), np.flatnonzero(fold_ids == fold))
          for fold in range(1, 6)]
y = train.target.to_numpy()
for fit, valid in splits:
    assert len(fit) and len(valid)
    assert set(train.iloc[fit].seller_id).isdisjoint(train.iloc[valid].seller_id)
    assert len(np.unique(y[valid])) == 2 and len(np.unique(y[fit])) == 2
versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'scipy': scipy.__version__, 'threadpoolctl': threadpoolctl.__version__}
print('Dane:', train.shape, test.shape)
print('Foldy:', fold_sha256)
print('Wersje:', versions)

Dane: (16000, 11) (10000, 10)
Foldy: c6a234966e2b3107f3c24e9d3aa0d565430027cb6e16212ebf612a993ccc6956
Wersje: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'threadpoolctl': '3.6.0'}


In [14]:
def features(df):
    x = df.copy()
    title = x.title.fillna('').astype(str).str.lower()
    desc = x.description.fillna('').astype(str).str.lower()
    x['text'] = title + ' ' + desc
    x['title_length'] = title.str.len()
    x['description_length'] = desc.str.len()
    # Missing observations remain missing; imputation is learned within each fold.
    for col in ['price_pln', 'shipping_weight_kg', 'pack_size']:
        x['log_' + col] = np.log1p(x[col].clip(lower=0))
    x['price_per_item'] = np.log1p(x.price_pln / x.pack_size.clip(lower=1))
    x['mass_per_item'] = np.log1p(x.shipping_weight_kg / x.pack_size.clip(lower=1))
    # Starter hint, not a guaranteed main-item count: free accessories and words
    # require a better parser. Evaluate improvements using held-out sellers.
    extracted = x.text.str.extract(r'(\d+)\s*pcs', expand=False)
    x['text_pack'] = pd.to_numeric(extracted, errors='coerce')
    x['pack_difference'] = (x.text_pack - x.pack_size).abs()
    # Useful text-category interactions; no vocabulary or statistics learned here.
    x['category_text'] = [
        ' '.join(f'{cat}__{token}' for token in re.findall(r'\w+', text))
        for cat, text in zip(x.declared_category.fillna('missing'), x.text)
    ]
    return x.replace({pd.NA: np.nan})

def make_model():
    numeric = ['log_price_pln', 'log_shipping_weight_kg', 'log_pack_size',
               'price_per_item', 'mass_per_item', 'text_pack', 'pack_difference',
               'title_length', 'description_length']
    categorical = ['declared_category', 'brand', 'condition']
    preprocessing = ColumnTransformer([
        ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=18000,
                                 sublinear_tf=True), 'text'),
        ('interactions', TfidfVectorizer(min_df=3, max_features=12000), 'category_text'),
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler(with_mean=False))]), numeric),
        ('categories', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                                 ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
    ], sparse_threshold=1.)
    return Pipeline([('preprocess', preprocessing),
                     ('model', LogisticRegression(C=2., max_iter=600, solver='liblinear',
                                                  random_state=SEED))])

import re
import math
import pandas as pd

NUMBER = r"(?:\d+|one|two|three|four|five)"
WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

PATTERNS = [
    rf"\b({NUMBER})\s+(?:individually\s+boxed\s+)?main\s+items?\b",
    rf"\bpack\s+of\s+({NUMBER})\b",
    rf"\bpackage\s+contains\s+({NUMBER})\s*(?:pcs|pieces|items?)\b",
]


def source_counts(value):
    # Brak tekstu traktujemy jako pusty opis.
    if pd.isna(value):
        text = ""
    else:
        text = str(value).lower()

    # Pomijamy fragmenty mówiące o gratisach lub akcesoriach.
    parts = re.split(r"[.;!?\n]", text)
    clean = " ".join(
        part for part in parts
        if not re.search(r"\b(?:free|bonus|accessor\w*)\b", part)
    )

    matches = [
        match
        for pattern in PATTERNS
        for match in re.findall(pattern, clean)
    ]

    # Jeśli nie znaleźliśmy dokładniejszego zapisu, szukamy np. "1 pcs".
    if not matches:
        matches = re.findall(
            rf"\b({NUMBER})\s*(?:pcs|pieces)\b",
            clean
        )

    return {
        WORDS[match] if match in WORDS else int(match)
        for match in matches
        if match in WORDS or int(match) > 0
    }


def quantity_hint(title, description):
    values = source_counts(title) | source_counts(description)

    # Przy braku ilości albo sprzecznych liczbach pozostawiamy brak.
    if len(values) == 1:
        return float(next(iter(values)))

    return float("nan")

## Jak powstaje cecha

Dla opisu `wireless headphones` w kategorii audio tworzymy tokeny
`audio__wireless` i `audio__headphones`. Ten sam opis zadeklarowany jako
kitchen tworzy inne tokeny. Model może uczyć się takich zestawień.

To nie jest reguła mówiąca, że każde nietypowe zestawienie jest błędem.
Niektóre produkty pasują do więcej niż jednej kategorii.

In [15]:
demo = pd.DataFrame([
    {'title': 'wireless headphones', 'description': '', 'declared_category': category,
     'brand': 'Noro', 'condition': 'new', 'price_pln': 100.0,
     'shipping_weight_kg': 0.4, 'pack_size': 1.0}
    for category in ['audio', 'kitchen']
])
print(features(demo)[['declared_category', 'category_text']].to_string(index=False))

declared_category                         category_text
            audio     audio__wireless audio__headphones
          kitchen kitchen__wireless kitchen__headphones


In [16]:
out = OUTPUT_ROOT / 'experiment_category'
out.mkdir(exist_ok=True)
x = features(train.drop(columns='target'))
x['text_pack'] = train.apply(lambda row: quantity_hint(row.title, row.description), axis=1)
x['pack_difference'] = (x.text_pack - x.pack_size).abs()
rows, probabilities = [], {}
for name, include_interactions in [('with_category_text', True), ('without_category_text', False)]:
    started = time.perf_counter()
    p = np.full(len(train), np.nan)
    row = {'configuration': name}
    for fold, (fit, valid) in enumerate(splits, 1):
        model = make_model()
        if not include_interactions:
            model.named_steps['preprocess'].set_params(interactions='drop')
        with threadpool_limits(limits=4):
            model.fit(x.iloc[fit], y[fit])
            p[valid] = model.predict_proba(x.iloc[valid])[:, 1]
        row[f'fold_{fold}'] = float(roc_auc_score(y[valid], p[valid]))
        print(f'{name} | fold {fold} | {row[f"fold_{fold}"]:.6f}', flush=True)
    assert np.isfinite(p).all()
    row['oof_roc_auc'] = float(roc_auc_score(y, p))
    row['seconds'] = time.perf_counter() - started
    rows.append(row)
    probabilities[name] = p
comparison = pd.DataFrame(rows)
comparison['delta_vs_reference'] = comparison.oof_roc_auc - rows[0]['oof_roc_auc']
comparison.to_csv(out / 'comparison.csv', index=False)
pd.DataFrame({'id': train.id, 'seller_id': train.seller_id, 'fold': fold_ids,
              'target': y, **probabilities}).to_csv(out / 'oof.csv', index=False)
fold_table.to_csv(out / 'folds.csv', index=False, lineterminator='\n')
metadata = {
    'hypothesis': 'Category-text interactions may help identify inconsistent listings.',
    'change': 'Only the category_text/interactions feature branch is removed.',
    'unchanged': 'Logistic regression family and all hyperparameters, training rows, quantity_v1 features, folds.',
    'classifier_parameters': make_model().named_steps['model'].get_params(),
    'preprocessing': 'TF-IDF vocabulary, imputation, scaling and encoding fitted within each training fold.',
    'seed': SEED, 'versions': versions, 'folds_sha256': fold_sha256,
    'train_sha256': EXPECTED_TRAIN_SHA256,
    'runtime_definition': 'Model creation, fit and validation inference; shared deterministic feature construction and output writing excluded.',
    'independent_final_estimate': False,
}
(out / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('\nPORÓWNANIE CECH:')
print(comparison.round(6).to_string(index=False))
delta = float(comparison.iloc[1].delta_vs_reference)
print(f'Zmiana po usunięciu grupy: {delta:+.6f}')
print('GOTOWE — eksperyment kategorii')

with_category_text | fold 1 | 0.711263
with_category_text | fold 2 | 0.695107
with_category_text | fold 3 | 0.725747
with_category_text | fold 4 | 0.723009
with_category_text | fold 5 | 0.729921
without_category_text | fold 1 | 0.612977
without_category_text | fold 2 | 0.612291
without_category_text | fold 3 | 0.608045
without_category_text | fold 4 | 0.626070
without_category_text | fold 5 | 0.628997

PORÓWNANIE CECH:
        configuration   fold_1   fold_2   fold_3   fold_4   fold_5  oof_roc_auc   seconds  delta_vs_reference
   with_category_text 0.711263 0.695107 0.725747 0.723009 0.729921     0.717692 14.835852            0.000000
without_category_text 0.612977 0.612291 0.608045 0.626070 0.628997     0.617712  9.651587           -0.099979
Zmiana po usunięciu grupy: -0.099979
GOTOWE — eksperyment kategorii


## Wnioski i ograniczenia

W zapisanym uruchomieniu Kaggle ze scikit-learn 1.6.1 usunięcie gałęzi
obniżyło pooled OOF ROC-AUC z 0.717692 do 0.617712, czyli o 0.099979.
Pogorszenie wystąpiło we wszystkich pięciu foldach.
Bieżące liczby i czasy znajdują się w tabeli powyżej.

W tym porównaniu informacja o zestawieniu kategorii ze słowami pomagała
regresji logistycznej. Wynik nie mówi, jaki rodzaj korupcji wystąpił
w konkretnej ofercie ani że taka sama zmiana pomoże każdej rodzinie modeli.

Foldy były używane do wyboru metody. Potrzebny byłby osobny nietknięty
zbiór sprzedawców lub nested CV do niezależnej oceny końcowej.